In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Public Test — Pseudo-labeling và Test-Time Augmentation

Reference thực hành. Dự đoán trước mỗi experiment, rồi ghi Result → Observation → Why.

## Data contract

Đây là pool synthetic được phép dùng trong lab. Trong thi thật, chỉ bật nhánh này nếu quy chế cho phép sử dụng input test để huấn luyện. Validation có nhãn luôn tách khỏi pool. TTA minh họa phản chiếu feature nhiễu x2, không phải công thức lật ảnh tùy ý.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def sample_inputs(n):
    """Return X (n, 2), with a sign-relevant first coordinate and nuisance second."""
    return rng.normal(size=(n, 2))

def label_inputs(X):
    """Generate labels (n,) only for the labeled training/validation fixtures."""
    return (X[:, 0] > 0).astype(int)

X_train = sample_inputs(40 if FAST else 80)
y_train = label_inputs(X_train)
X_val = sample_inputs(160 if FAST else 320)
y_val = label_inputs(X_val)
X_public = sample_inputs(200 if FAST else 500)
# WHY: no y_public is created or consulted.
assert set(np.unique(y_train)) == {0, 1}

def build_model():
    """Return a fresh numeric binary classifier."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500, random_state=42))

baseline = build_model().fit(X_train, y_train)
public_prob = baseline.predict_proba(X_public)
assert baseline.classes_.tolist() == [0, 1]


## Worked Example bằng code

In [ ]:
toy = np.array([[0.1, 0.9], [0.55, 0.45]])
assert (toy.max(axis=1) >= 0.85).tolist() == [True, False]
assert np.allclose(np.mean([0.8, 0.4]), 0.6)


## Experiment 1 — Một vòng pseudo-label

**Hypothesis:** tau cao chọn ít mẫu hơn nhưng không đảm bảo F1 tăng. Dự đoán counts trước khi chạy. Baseline sinh probability một lần; mỗi candidate được fit mới, không lặp tự huấn luyện nhiều vòng.

In [ ]:
def one_round(threshold, allowed=True):
    """Return model and selected count; never use validation labels for training."""
    mask = public_prob.max(axis=1) >= threshold
    if not allowed or not mask.any():
        return baseline, 0
    pseudo = baseline.classes_[public_prob[mask].argmax(axis=1)]
    X_aug = np.concatenate([X_train, X_public[mask]], axis=0)
    y_aug = np.concatenate([y_train, pseudo], axis=0)
    return build_model().fit(X_aug, y_aug), int(mask.sum())

def score(model, X):
    """Evaluate Macro F1 against held-out y_val; X must correspond to those rows."""
    return f1_score(y_val, model.predict(X), labels=[0, 1], average='macro', zero_division=0)

rows = [{'method': 'baseline', 'threshold': None, 'selected': 0,
         'macro_f1': score(baseline, X_val)}]
counts = []
for threshold in [0.70, 0.85, 0.95]:
    candidate, count = one_round(threshold)
    counts.append(count)
    rows.append({'method': 'pseudo_one_round', 'threshold': threshold,
                 'selected': count, 'macro_f1': score(candidate, X_val)})
assert counts == sorted(counts, reverse=True)
empty_model, empty_count = one_round(1.01)
blocked_model, blocked_count = one_round(0.7, allowed=False)
assert empty_model is baseline and empty_count == 0
assert blocked_model is baseline and blocked_count == 0


## Experiment 2 — Test-Time Augmentation

**Hypothesis:** phản chiếu x2 loại bớt ảnh hưởng của nhiễu ngẫu nhiên mà model học. Label phụ thuộc x1 nên invariant với phép này. **Observation/Why:** ghi F1 thực tế; không tự kết luận mọi phép lật đều hợp lệ.

In [ ]:
def reflect_nuisance(X):
    """Return (n, 2) after negating only the nuisance coordinate."""
    reflected = X.copy()
    reflected[:, 1] *= -1
    return reflected

assert np.array_equal(label_inputs(X_val), label_inputs(reflect_nuisance(X_val)))
views = np.stack([baseline.predict_proba(X_val),
                  baseline.predict_proba(reflect_nuisance(X_val))], axis=0)
tta_prob = views.mean(axis=0)
assert views.shape == (2, len(X_val), 2)
assert np.allclose(tta_prob.sum(axis=1), 1)
tta_score = f1_score(y_val, (tta_prob[:, 1] >= 0.5).astype(int),
                     labels=[0, 1], average='macro', zero_division=0)
rows.append({'method': 'baseline_TTA', 'threshold': None, 'selected': 0, 'macro_f1': tta_score})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison.to_csv(OUT / 'public_test_experiments.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(np.arange(len(rows)), comparison['macro_f1'], label='Validation Macro F1')
ax.set_xticks(np.arange(len(rows)), ['base', 'tau .70', 'tau .85', 'tau .95', 'TTA'])
ax.set(title='Pseudo-label and TTA comparison', xlabel='Candidate', ylabel='Macro F1', ylim=(0, 1))
ax.legend()
plt.show()
plt.close(fig)


## Quyết định & Mastery

Ghi candidate nào giữ lại, số mẫu nhãn giả theo class, và giả định TTA. Validation dùng nhiều lần là development set; nếu cần ước lượng cuối độc lập, dành một labeled hold-out khác trước mọi lựa chọn. Làm U-1/I-1/E-1. Không dùng bảng này để suy ra quyền sử dụng dữ liệu cuộc thi.